In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import json
import os

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
IMG_DIR = os.path.join(BASE_DIR, 'data/images')
TRAIN_JSON = os.path.join(BASE_DIR, 'data/labels', 'train_map.json')
TEST_JSON = os.path.join(BASE_DIR, 'data/labels', 'test_map.json') 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Praca na: {DEVICE}")

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.4, 1.0)), 
    transforms.RandomPerspective(distortion_scale=0.3, p=0.5),
    
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1), 
    
    transforms.GaussianBlur(kernel_size=(3, 5), sigma=(0.1, 2.0)),
    
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class BalancedUrbanDataset(Dataset):
    def __init__(self, json_file, img_dir, samples_per_class=100, transform=None):
        with open(json_file, 'r') as f:
            full_map = json.load(f)
        
        self.img_dir = img_dir
        self.transform = transform
        
        cat_to_paths = {}
        for path, cat in full_map.items():
            if cat not in cat_to_paths: cat_to_paths[cat] = []
            cat_to_paths[cat].append(path)
        
        self.categories = sorted(list(cat_to_paths.keys()))
        self.class_to_idx = {cat: i for i, cat in enumerate(self.categories)}
        
        self.final_samples = []
        for cat in self.categories:
            paths = cat_to_paths.get(cat, [])
            num_to_take = min(samples_per_class, len(paths))
            for i in range(num_to_take):
                self.final_samples.append((paths[i], self.class_to_idx[cat]))
                
        print(f"Załadowano {len(self.final_samples)} obrazów z {os.path.basename(json_file)}")

    def __len__(self):
        return len(self.final_samples)

    def __getitem__(self, idx):
        path, label = self.final_samples[idx]
        img = Image.open(os.path.join(self.img_dir, path)).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

def get_model(num_classes):
    model = models.resnet18(weights='IMAGENET1K_V1')
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model.to(DEVICE)

train_dataset = BalancedUrbanDataset(TRAIN_JSON, IMG_DIR, samples_per_class=200, transform=train_transforms)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,pin_memory=True)

test_dataset = BalancedUrbanDataset(TEST_JSON, IMG_DIR, samples_per_class=10000, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

model = get_model(len(train_dataset.categories))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001, weight_decay=0.01)

print("\nRozpoczynam trening...")
best_accuracy = 0.0
for epoch in range(15):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    test_acc = 100 * correct / total
    print(f"Epoka [{epoch+1}/10] | Loss: {running_loss/len(train_loader):.4f} | Test Accuracy: {test_acc:.2f}%")
    if test_acc > best_accuracy:
        best_accuracy = test_acc
        torch.save(model.state_dict(), "Best.pth")
        print(f"  >>> Zapisano nowy najlepszy model! (Acc: {best_accuracy:.2f}%)")

torch.save(model.state_dict(), "LastOne.pth")
print("\nModel zapisany.")

Praca na: cuda
Załadowano 1200 obrazów z train_map.json
Załadowano 2246 obrazów z test_map.json

Rozpoczynam trening...
Epoka [1/10] | Loss: 1.5541 | Test Accuracy: 56.86%
  >>> Zapisano nowy najlepszy model! (Acc: 56.86%)
Epoka [2/10] | Loss: 1.0184 | Test Accuracy: 67.05%
  >>> Zapisano nowy najlepszy model! (Acc: 67.05%)
Epoka [3/10] | Loss: 0.7751 | Test Accuracy: 70.75%
  >>> Zapisano nowy najlepszy model! (Acc: 70.75%)
Epoka [4/10] | Loss: 0.6359 | Test Accuracy: 75.60%
  >>> Zapisano nowy najlepszy model! (Acc: 75.60%)
Epoka [5/10] | Loss: 0.5382 | Test Accuracy: 78.05%
  >>> Zapisano nowy najlepszy model! (Acc: 78.05%)
Epoka [6/10] | Loss: 0.4748 | Test Accuracy: 81.52%
  >>> Zapisano nowy najlepszy model! (Acc: 81.52%)
Epoka [7/10] | Loss: 0.4347 | Test Accuracy: 78.81%
Epoka [8/10] | Loss: 0.3776 | Test Accuracy: 76.85%
Epoka [9/10] | Loss: 0.3838 | Test Accuracy: 79.96%
Epoka [10/10] | Loss: 0.3420 | Test Accuracy: 79.25%
Epoka [11/10] | Loss: 0.3468 | Test Accuracy: 81.26%
